In [5]:
# 1. Target the zip file inside sample_data and extract it locally
# The -o flag forces an automatic overwrite
!unzip -o -q sample_data/archive.zip -d sample_data/

import torch
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os
from PIL import Image
from google.colab import files
import io

print("Initializing Data Pipeline...")

# 2. Dynamic Data Augmentation + ImageNet Normalization Matrix
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Explicitly point the engine to the sample_data folder
if os.path.exists('sample_data/brain_tumor_dataset'):
    data_dir = 'sample_data/brain_tumor_dataset'
else:
    data_dir = 'sample_data'

# 4. Load the tensors
dataset = datasets.ImageFolder(root=data_dir, transform=train_transform)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Loaded {len(dataset)} augmented spatial matrices. Class mapping: {dataset.class_to_idx}")

Initializing Data Pipeline...
Loaded 253 augmented spatial matrices. Class mapping: {'no': 0, 'yes': 1}


In [6]:
print("Downloading Pre-Trained ResNet-18 Architecture...")

# 1. Load the industry-standard ResNet-18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# 2. Freeze the core feature-extraction layers
for param in model.parameters():
    param.requires_grad = False

# 3. Rip out the final classification layer and replace it with our 2-class logic
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

# 4. Push to Cloud GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Architecture locked and loaded onto: {device}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 163MB/s]


Architecture locked and loaded onto: cuda


In [7]:
# 1. Rig the penalty matrix to punish False Negatives severely
# Assuming Class 0 is 'no' and Class 1 is 'yes'
class_weights = torch.tensor([1.0, 3.0]).to(device)

# 2. Inject the weights into the CrossEntropy engine
criterion = nn.CrossEntropyLoss(weight=class_weights)

# 3. ONLY optimize the newly attached final layer
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

epochs = 10

print("Initiating Weighted Fine-Tuning Sequence...")

for epoch in range(epochs):
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_accuracy = (correct_predictions / total_samples) * 100
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Accuracy: {epoch_accuracy:.2f}%")

# Hardcode the final weights
torch.save(model.state_dict(), 'resnet18_tumor_weights.pth')
print("Training complete. Weights saved to resnet18_tumor_weights.pth")

Initiating Weighted Fine-Tuning Sequence...
Epoch 1/10 | Loss: 0.5267 | Accuracy: 60.87%
Epoch 2/10 | Loss: 0.4339 | Accuracy: 64.03%
Epoch 3/10 | Loss: 0.3805 | Accuracy: 62.06%
Epoch 4/10 | Loss: 0.3676 | Accuracy: 65.61%
Epoch 5/10 | Loss: 0.3264 | Accuracy: 70.36%
Epoch 6/10 | Loss: 0.3284 | Accuracy: 72.33%
Epoch 7/10 | Loss: 0.3205 | Accuracy: 73.52%
Epoch 8/10 | Loss: 0.2730 | Accuracy: 75.10%
Epoch 9/10 | Loss: 0.2641 | Accuracy: 79.45%
Epoch 10/10 | Loss: 0.2421 | Accuracy: 80.63%
Training complete. Weights saved to resnet18_tumor_weights.pth


In [18]:
print("Awaiting target MRI scan for Final Inference...")
uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))

    # Standardize and Normalize the incoming geometry
    inference_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    raw_image = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')
    image_tensor = inference_transform(raw_image).unsqueeze(0).to(device)

    # Lock the model and calculate probabilities
    model.eval()
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = F.softmax(output, dim=1)[0]

    # Extract highest value
    confidence, predicted_class = torch.max(probabilities, 0)

    # Map output (Assuming 'no': 0, 'yes': 1 based on previous checks)
    prediction = "YES" if predicted_class.item() == 1 else "NO"
    conf_score = confidence.item() * 100

    print(f"\n==============================")
    print(f"--- CLINICAL INFERENCE ---")
    print(f"Tumor Detected: {prediction}")
    print(f"Confidence:     {conf_score:.2f}%")
    print(f"==============================")

Awaiting target MRI scan for Final Inference...


Saving YES.jpg to YES.jpg

--- CLINICAL INFERENCE ---
Tumor Detected: YES
Confidence:     98.78%


In [9]:
import torch.optim as optim
from torch.optim import lr_scheduler

print("Initiating Stage 2: Deep Architecture Fine-Tuning...")

# 1. Dial back the paranoia weight to balance the engine
class_weights = torch.tensor([1.0, 1.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# 2. Unfreeze the last convolutional block of ResNet (layer4)
for param in model.layer4.parameters():
    param.requires_grad = True

# 3. Create a dual-speed optimizer
# The deep layers learn SLOW (1e-5), the final layer learns FAST (1e-4)
optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}
])

# 4. Add a Learning Rate Scheduler (decays LR by 10% every 5 epochs)
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

epochs = 15 # Pushing more epochs because the LR actively decays

for epoch in range(epochs):
    model.train() # Ensure dropout/batchnorm are in training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    scheduler.step() # Trigger the mathematical decay

    epoch_accuracy = (correct_predictions / total_samples) * 100
    current_lr = optimizer.param_groups[0]['lr'] # Track the deep layer learning rate

    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Acc: {epoch_accuracy:.2f}% | Deep LR: {current_lr}")

# Overwrite the old weights with the high-accuracy ones
torch.save(model.state_dict(), 'resnet18_tumor_weights_v2.pth')
print("\nStage 2 Complete. Deep weights saved to resnet18_tumor_weights_v2.pth")

Initiating Stage 2: Deep Architecture Fine-Tuning...
Epoch 1/15 | Loss: 0.3269 | Acc: 84.98% | Deep LR: 1e-05
Epoch 2/15 | Loss: 0.2757 | Acc: 88.14% | Deep LR: 1e-05
Epoch 3/15 | Loss: 0.2370 | Acc: 93.28% | Deep LR: 1e-05
Epoch 4/15 | Loss: 0.2490 | Acc: 90.51% | Deep LR: 1e-05
Epoch 5/15 | Loss: 0.2034 | Acc: 92.49% | Deep LR: 1.0000000000000002e-06
Epoch 6/15 | Loss: 0.2035 | Acc: 91.70% | Deep LR: 1.0000000000000002e-06
Epoch 7/15 | Loss: 0.1840 | Acc: 94.47% | Deep LR: 1.0000000000000002e-06
Epoch 8/15 | Loss: 0.2058 | Acc: 94.07% | Deep LR: 1.0000000000000002e-06
Epoch 9/15 | Loss: 0.1830 | Acc: 94.47% | Deep LR: 1.0000000000000002e-06
Epoch 10/15 | Loss: 0.1840 | Acc: 93.68% | Deep LR: 1.0000000000000002e-07
Epoch 11/15 | Loss: 0.1801 | Acc: 94.47% | Deep LR: 1.0000000000000002e-07
Epoch 12/15 | Loss: 0.1893 | Acc: 94.07% | Deep LR: 1.0000000000000002e-07
Epoch 13/15 | Loss: 0.1822 | Acc: 93.68% | Deep LR: 1.0000000000000002e-07
Epoch 14/15 | Loss: 0.1859 | Acc: 92.89% | Deep L

In [19]:
from google.colab import files

print("Initiating download sequence for Stage 2 weights...")
files.download('resnet18_tumor_weights_v2.pth')

Initiating download sequence for Stage 2 weights...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>